In [10]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

data_path = r"G:\My Drive\New_Working_Files\KNIME\CTU_2026\Kapture_Reports\Outputs\PL_Outputs.xlsx"
df = pd.read_excel(data_path)
print(df.head(2))

  crm_flag       Date     Month  Week                            case_id  \
0     SRMS 2026-01-21   January     4  FLRM-65P5D8-VF6HZV-GMRX9G-ZI4ACFK   
1     SRMS 2026-02-03  February     6  FLRM-6HNUGF-TG7UDG-KS37A8-P3OTYWK   

                     CallTypeDetails     RegisteredBy       Source  \
0  Fake Status Updated By Technician  IENODJV-T301591  inboundCall   
1        To Check Status Of The Call      DRITM.19463  inboundCall   

                                         Description complaint_category  \
0  undefined - undefined - cx want to know the st...             REPAIR   
1  undefined - undefined - Voc customer want to k...             REPAIR   

  Ticket Status Ticket SubStatus client_order_id brand_name call_status  \
0     COMPLETED   NOT_APPLICABLE             NaN        NaN   COMPLETED   
1     CANCELLED    VAS_INITIATED             NaN        NaN   CANCELLED   

             call_cancelled_date                registered_date        EmpID  \
0                           

In [12]:
def clean_text(text):
    text=str(text).lower().replace('undefined - undefined -','')
    return text.strip()

df['Clean_Text'] = df['Description'].apply(clean_text)

vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(df['Clean_Text'])

num_clusters = 5
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
kmeans.fit(X)

df['Discovered_Cluster'] = kmeans.labels_
print("/n---- What are these clusters about?----")
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]
terms = vectorizer.get_feature_names_out()

for i in range(num_clusters):
    top_words = [terms[ind] for ind in order_centroids[i, :4]]
    print(f"Cluster {i} Keywords: {', '.join(top_words)}")

print("\n ---CATEGORIZED TICKETS ----")
print(df[['Clean_Text', 'Discovered_Cluster']].sort_values(by='Discovered_Cluster'))
                                           
             

/n---- What are these clusters about?----
Cluster 0 Keywords: na, blank, cx, drop
Cluster 1 Keywords: nan, zrz1vtj, zqz0di, zpjh83b
Cluster 2 Keywords: cx, wait, st, status
Cluster 3 Keywords: cx, product, service, flipkart
Cluster 4 Keywords: concerned, team, allow, escalated

 ---CATEGORIZED TICKETS ----
                                               Clean_Text  Discovered_Cluster
54913   na  - na - blank and drop call. i call back bu...                   0
54908                           na - na - blank call drop                   0
54907   na - na - agent was transferring the call but ...                   0
54929   na - na - cx- come and say he want to know tha...                   0
54928   na - na - cx- come and say he want to know tha...                   0
...                                                   ...                 ...
136177  water purifier - kenstar - your issue has been...                   4
134110  inductioncooktop - sansui  - cx wants to know ...           